In [ ]:
"""
=============================================================================
FIGURE 1.4 - ЧАСТЬ 3: ФИТТИНГ МОДЕЛИ РАЗРЕШЕНИЯ
=============================================================================
Цель: Найти параметры разрешения детектора (a, b, c)

Модель: (σ_E/E₀)² = (a/√E₀)² + (b/E₀)² + c²

Компоненты:
  - a: стохастический член (статистические флуктуации)
  - b: шумовой член (электронный шум)
  - c: постоянный член (систематические эффекты)

Метод: Нелинейный Least Squares через iminuit
"""

print("="*70)
print("ШАГ 3: ФИТТИНГ ПАРАМЕТРОВ РАЗРЕШЕНИЯ (a, b, c)")
print("="*70)
print("\nМодель: σ_E/E₀ = √[(a/√E₀)² + (b/E₀)² + c²]")
print("\nЗапуск минимизации iminuit...")

# Запускаем фиттинг разрешения
res_minuit, res_params, res_errors_dict = fitting.fit_resolution_parameters(
    E0_values,     # x-данные: истинные энергии
    stds,          # y-данные: стандартные отклонения
    std_errors     # ошибки на y: standard errors of std
)

# Извлекаем результаты
a_fit = res_params['a']
b_fit = res_params['b']
c_fit = res_params['c']
a_err = res_errors_dict['a']
b_err = res_errors_dict['b']
c_err = res_errors_dict['c']

# Выводим результаты
print("\n" + "-"*70)
print("РЕЗУЛЬТАТЫ ФИТА:")
print("-"*70)
print(f"a (stochastic) = {a_fit:.6f} ± {a_err:.6f}")
print(f"b (noise)      = {b_fit:.6f} ± {b_err:.6f}")
print(f"c (constant)   = {c_fit:.6f} ± {c_err:.6f}")

# Качество фита
print("\n" + "-"*70)
print("КАЧЕСТВО ФИТА:")
print("-"*70)
print(f"χ² = {res_minuit.fval:.2f}")
print(f"ndf = {res_minuit.ndof}")
print(f"χ²/ndf = {res_minuit.fval/res_minuit.ndof:.2f}")
print(f"Статус: {'✓ УСПЕШНО' if res_minuit.valid else '✗ ОШИБКА'}")

# Анализ вкладов при разных энергиях
print("\n" + "-"*70)
print("ВКЛАДЫ КОМПОНЕНТ ПРИ РАЗНЫХ ЭНЕРГИЯХ:")
print("-"*70)

for E0 in [20, 50, 80]:  # Примеры низкой, средней и высокой энергии
    # Вычисляем вклад каждого члена
    term_a = (a_fit / np.sqrt(E0))**2
    term_b = (b_fit / E0)**2
    term_c = c_fit**2
    total = np.sqrt(term_a + term_b + term_c)
    
    # Процентный вклад
    contrib_a = term_a / (term_a + term_b + term_c) * 100
    contrib_b = term_b / (term_a + term_b + term_c) * 100
    contrib_c = term_c / (term_a + term_b + term_c) * 100
    
    print(f"\nE₀ = {E0} GeV: σ/E₀ = {total:.4f}")
    print(f"  Stochastic (a): {contrib_a:5.1f}%")
    print(f"  Noise (b):      {contrib_b:5.1f}%")
    print(f"  Constant (c):   {contrib_c:5.1f}%")

print("\n" + "="*70)

In [ ]:
"""
=============================================================================
FIGURE 1.4 - ЧАСТЬ 2: ФИТТИНГ МОДЕЛИ СРЕДНЕГО (μ = λ·E₀ + Δ)
=============================================================================
Цель: Найти оптимальные значения λ (lambda) и Δ (Delta)

Модель: μ_E(E₀) = λ·E₀ + Δ
  - λ (lambda): масштабный коэффициент (ideally ≈ 1.0)
  - Δ (Delta):  смещение/offset (ideally ≈ 0.0)

Метод: Weighted Least Squares через iminuit.LeastSquares
"""

from s1_sol import fitting

print("="*70)
print("ШАГ 2: ФИТТИНГ ПАРАМЕТРОВ СРЕДНЕГО ЗНАЧЕНИЯ (λ, Δ)")
print("="*70)
print("\nМодель: μ_E = λ·E₀ + Δ")
print("\nЗапуск минимизации iminuit...")

# Запускаем фиттинг с использованием iminuit
mean_minuit, mean_params, mean_errors_dict = fitting.fit_mean_parameters(
    E0_values,     # x-данные: истинные энергии
    means,         # y-данные: измеренные средние
    mean_errors    # ошибки на y: standard errors
)

# Извлекаем результаты
lambda_fit = mean_params['lambda']
Delta_fit = mean_params['Delta']
lambda_err = mean_errors_dict['lambda']
Delta_err = mean_errors_dict['Delta']

# Выводим результаты
print("\n" + "-"*70)
print("РЕЗУЛЬТАТЫ ФИТА:")
print("-"*70)
print(f"λ (lambda) = {lambda_fit:.6f} ± {lambda_err:.6f}")
print(f"Δ (Delta)  = {Delta_fit:.4f} ± {Delta_err:.4f} GeV")

# Проверяем качество фита
print("\n" + "-"*70)
print("КАЧЕСТВО ФИТА:")
print("-"*70)
print(f"χ² = {mean_minuit.fval:.2f}")
print(f"ndf (degrees of freedom) = {mean_minuit.ndof}")
print(f"χ²/ndf = {mean_minuit.fval/mean_minuit.ndof:.2f}")
print(f"Статус: {'✓ УСПЕШНО' if mean_minuit.valid else '✗ ОШИБКА'}")

# Интерпретация результатов
print("\n" + "-"*70)
print("ФИЗИЧЕСКАЯ ИНТЕРПРЕТАЦИЯ:")
print("-"*70)

# Проверка λ
if abs(lambda_fit - 1.0) < 0.01:
    print(f"✓ λ ≈ 1.000: Детектор хорошо откалиброван")
else:
    deviation = (lambda_fit - 1.0) * 100
    print(f"⚠ λ = {lambda_fit:.4f}: Отклонение от идеала на {deviation:+.2f}%")

# Проверка Δ
if abs(Delta_fit) < 0.5:
    print(f"✓ Δ ≈ 0: Систематическое смещение минимально")
else:
    print(f"⚠ Δ = {Delta_fit:.3f} GeV: Присутствует заметное смещение")

print("\n" + "="*70)

In [ ]:
"""
=============================================================================
FIGURE 1.4 - ЧАСТЬ 3: ФИТТИНГ МОДЕЛИ РАЗРЕШЕНИЯ
=============================================================================
Цель: Найти параметры разрешения детектора (a, b, c)

Модель: (σ_E/E₀)² = (a/√E₀)² + (b/E₀)² + c²

Компоненты:
  - a: стохастический член (статистические флуктуации)
  - b: шумовой член (электронный шум)
  - c: постоянный член (систематические эффекты)

Метод: Нелинейный Least Squares через iminuit
"""

print("="*70)
print("ШАГ 3: ФИТТИНГ ПАРАМЕТРОВ РАЗРЕШЕНИЯ (a, b, c)")
print("="*70)
print("\nМодель: σ_E/E₀ = √[(a/√E₀)² + (b/E₀)² + c²]")
print("\nЗапуск минимизации iminuit...")

# Запускаем фиттинг разрешения
res_minuit, res_params, res_errors_dict = fitting.fit_resolution_parameters(
    E0_values,     # x-данные: истинные энергии
    stds,          # y-данные: стандартные отклонения
    std_errors     # ошибки на y: standard errors of std
)

# Извлекаем результаты
a_fit = res_params['a']
b_fit = res_params['b']
c_fit = res_params['c']
a_err = res_errors_dict['a']
b_err = res_errors_dict['b']
c_err = res_errors_dict['c']

# Выводим результаты
print("\n" + "-"*70)
print("РЕЗУЛЬТАТЫ ФИТА:")
print("-"*70)
print(f"a (stochastic) = {a_fit:.6f} ± {a_err:.6f}")
print(f"b (noise)      = {b_fit:.6f} ± {b_err:.6f}")
print(f"c (constant)   = {c_fit:.6f} ± {c_err:.6f}")

# Качество фита
print("\n" + "-"*70)
print("КАЧЕСТВО ФИТА:")
print("-"*70)
print(f"χ² = {res_minuit.fval:.2f}")
print(f"ndf = {res_minuit.ndof}")
print(f"χ²/ndf = {res_minuit.fval/res_minuit.ndof:.2f}")
print(f"Статус: {'✓ УСПЕШНО' if res_minuit.valid else '✗ ОШИБКА'}")

# Анализ вкладов при разных энергиях
print("\n" + "-"*70)
print("ВКЛАДЫ КОМПОНЕНТ ПРИ РАЗНЫХ ЭНЕРГИЯХ:")
print("-"*70)

for E0 in [20, 50, 80]:  # Примеры низкой, средней и высокой энергии
    # Вычисляем вклад каждого члена
    term_a = (a_fit / np.sqrt(E0))**2
    term_b = (b_fit / E0)**2
    term_c = c_fit**2
    total = np.sqrt(term_a + term_b + term_c)
    
    # Процентный вклад
    contrib_a = term_a / (term_a + term_b + term_c) * 100
    contrib_b = term_b / (term_a + term_b + term_c) * 100
    contrib_c = term_c / (term_a + term_b + term_c) * 100
    
    print(f"\nE₀ = {E0} GeV: σ/E₀ = {total:.4f}")
    print(f"  Stochastic (a): {contrib_a:5.1f}%")
    print(f"  Noise (b):      {contrib_b:5.1f}%")
    print(f"  Constant (c):   {contrib_c:5.1f}%")

print("\n" + "="*70)

In [ ]:
"""
=============================================================================
FIGURE 1.4 - ЧАСТЬ 4 (OPTIONAL): BOOTSTRAP АНАЛИЗ
=============================================================================
Цель: Оценить робастность фита и получить error bands

Метод:
1. Для каждого E₀: resample данные с заменой (bootstrap)
2. Пересчитать μ̂ и σ̂ для каждой bootstrap выборки
3. Повторить фиттинг
4. Повторить N раз (обычно 100-1000)
5. Получить распределение параметров

ВНИМАНИЕ: Это может занять 1-2 минуты!
"""

# Флаг: запускать ли bootstrap (установите False чтобы пропустить)
RUN_BOOTSTRAP = True
N_BOOTSTRAP = 100  # Количество bootstrap samples

if RUN_BOOTSTRAP:
    print("="*70)
    print(f"ШАГ 4: BOOTSTRAP АНАЛИЗ ({N_BOOTSTRAP} samples)")
    print("="*70)
    print("\n⏳ Это займет ~1-2 минуты...")
    print("Прогресс будет показан для каждых 10 samples\n")
    
    # Запускаем bootstrap
    bootstrap_results = fitting.bootstrap_fit(grouped_data, n_bootstrap=N_BOOTSTRAP)
    
    # Статистика bootstrap результатов
    print("\n" + "-"*70)
    print("BOOTSTRAP РЕЗУЛЬТАТЫ:")
    print("-"*70)
    
    params_names = ['lambda', 'Delta', 'a', 'b', 'c']
    params_labels = ['λ (lambda)', 'Δ (Delta)', 'a', 'b', 'c']
    
    for name, label in zip(params_names, params_labels):
        values = bootstrap_results[name]
        mean_boot = np.mean(values)
        std_boot = np.std(values)
        
        print(f"\n{label}:")
        print(f"  Mean:   {mean_boot:.6f}")
        print(f"  StdDev: {std_boot:.6f}")
        print(f"  Range:  [{np.min(values):.6f}, {np.max(values):.6f}]")
    
    # Сравнение с исходным фитом
    print("\n" + "-"*70)
    print("СРАВНЕНИЕ: FIT vs BOOTSTRAP")
    print("-"*70)
    
    print(f"\nλ:  {lambda_fit:.6f} ± {lambda_err:.6f} (fit)")
    print(f"    {np.mean(bootstrap_results['lambda']):.6f} ± {np.std(bootstrap_results['lambda']):.6f} (bootstrap)")
    
    print(f"\nΔ:  {Delta_fit:.4f} ± {Delta_err:.4f} (fit)")
    print(f"    {np.mean(bootstrap_results['Delta']):.4f} ± {np.std(bootstrap_results['Delta']):.4f} (bootstrap)")
    
    print(f"\na:  {a_fit:.6f} ± {a_err:.6f} (fit)")
    print(f"    {np.mean(bootstrap_results['a']):.6f} ± {np.std(bootstrap_results['a']):.6f} (bootstrap)")
    
    print(f"\nb:  {b_fit:.6f} ± {b_err:.6f} (fit)")
    print(f"    {np.mean(bootstrap_results['b']):.6f} ± {np.std(bootstrap_results['b']):.6f} (bootstrap)")
    
    print(f"\nc:  {c_fit:.6f} ± {c_err:.6f} (fit)")
    print(f"    {np.mean(bootstrap_results['c']):.6f} ± {np.std(bootstrap_results['c']):.6f} (bootstrap)")
    
    print("\n" + "="*70)
    print("✓ Bootstrap завершен!")
    print("="*70)
    
else:
    print("="*70)
    print("ШАГ 4: BOOTSTRAP ПРОПУЩЕН (установите RUN_BOOTSTRAP = True чтобы запустить)")
    print("="*70)
    bootstrap_results = None

In [ ]:
"""
=============================================================================
FIGURE 1.4 - ЧАСТЬ 5: ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
=============================================================================
Создаем финальный график с:
  - Левая панель: μ̂ - E₀ с fitted линией
  - Правая панель: σ̂/E₀ с fitted кривой
  - Error bands из bootstrap (если доступны)
"""

import matplotlib.pyplot as plt

print("="*70)
print("ШАГ 5: СОЗДАНИЕ FIGURE 1.4")
print("="*70)

# Создаем график
fig, ax = fitting.plot_fits_with_bands(
    E0_values,          # x-координаты
    means,              # данные для левого графика
    mean_errors,        # ошибки для левого графика
    stds,               # данные для правого графика
    std_errors,         # ошибки для правого графика
    mean_params,        # параметры фита (λ, Δ)
    res_params,         # параметры фита (a, b, c)
    bootstrap_results   # результаты bootstrap (или None)
)

# Сохраняем в файл
from s1_sol import plotting
plotting.save_figure(fig, 'Figure1.4.pdf')

print("\n✓ График создан и сохранен: figs/Figure1.4.pdf")
print("="*70)

plt.show()

In [ ]:
"""
=============================================================================
FIGURE 1.4 - ЧАСТЬ 6: СОХРАНЕНИЕ РЕЗУЛЬТАТОВ ДЛЯ results.json
=============================================================================
Сохраняем оценки параметров в формате для финального results.json
"""

print("="*70)
print("ШАГ 6: ПОДГОТОВКА ДАННЫХ ДЛЯ results.json")
print("="*70)

# Формируем словарь с результатами sample estimates
sample_est_results = {
    'values': {
        'lb': float(lambda_fit),
        'dE': float(Delta_fit),
        'a': float(a_fit),
        'b': float(b_fit),
        'c': float(c_fit)
    },
    'errors': {
        'lb': float(lambda_err),
        'dE': float(Delta_err),
        'a': float(a_err),
        'b': float(b_err),
        'c': float(c_err)
    }
}

# Выводим финальную сводку
print("\nФИНАЛЬНЫЕ ОЦЕНКИ ПАРАМЕТРОВ (Sample Estimates Method):")
print("-"*70)
print("Параметр    | Значение    | Ошибка")
print("-"*70)
print(f"λ (lambda)  | {sample_est_results['values']['lb']:10.6f} | {sample_est_results['errors']['lb']:.6f}")
print(f"Δ (Delta)   | {sample_est_results['values']['dE']:10.4f} | {sample_est_results['errors']['dE']:.4f}")
print(f"a           | {sample_est_results['values']['a']:10.6f} | {sample_est_results['errors']['a']:.6f}")
print(f"b           | {sample_est_results['values']['b']:10.6f} | {sample_est_results['errors']['b']:.6f}")
print(f"c           | {sample_est_results['values']['c']:10.6f} | {sample_est_results['errors']['c']:.6f}")
print("-"*70)

print("\n✓ Результаты готовы для записи в results.json")
print("  (это будет сделано в финальной ячейке после всех трех методов)")
print("="*70)